# First queries

Install with `pip install "krishiv[pyarrow]"` (Python 3.11 or newer). The data is generated by `docs/datasets/generate.py`.

In [ ]:
import pathlib, sys, tempfile
sys.path.insert(0, "../datasets")  # docs/notebooks -> docs/datasets
import generate
import krishiv
from krishiv.sql import functions as F

data = pathlib.Path(tempfile.mkdtemp())
generate.generate(data, orders=20_000, customers=500)
spark = krishiv.session()
orders = spark.read.parquet(str(data / "orders.parquet"))
customers = spark.read.parquet(str(data / "customers.parquet"))
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

In [ ]:
spark.sql("SELECT count(*) AS n, round(sum(amount), 2) AS total FROM orders").show()

In [ ]:
top = spark.sql("SELECT customer_id, sum(amount) AS spent FROM orders GROUP BY customer_id ORDER BY spent DESC LIMIT 5")
assert top.count() == 5
top.show()

In [ ]:
big = orders.filter(F.col("amount") > 400).count()
assert 0 < big < 20_000
big